In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import joblib
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

RANDOM_STATE = 42

base_dir = "/content/drive/MyDrive/fraud processed data"
train_path = os.path.join(base_dir, "train_processed.csv")
test_path = os.path.join(base_dir, "test_processed.csv")

results_dir = os.path.join(base_dir, "rf_8_experiments_after_smote_cv3")
os.makedirs(results_dir, exist_ok=True)

print("Reading data...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]

X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nApplying SMOTE to training data only...")
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nOriginal y_train distribution:")
print(pd.Series(y_train).value_counts())

print("\nAfter SMOTE y_train distribution:")
print(pd.Series(y_train_smote).value_counts())

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

"""
تشغيل تجربة تدريب وتقييم نموذج Random Forest باستخدام مجموعة من المعاملات (Hyperparameters).

آلية العمل:
1. إنشاء مجلد خاص بالتجربة لحفظ جميع النتائج والملفات الناتجة.
2. التحقق مما إذا كانت التجربة قد نُفذت سابقاً من خلال وجود ملف metrics.json،
   وفي هذه الحالة يتم تحميل النتائج السابقة وتجنب إعادة التدريب.
3. إنشاء نموذج Random Forest باستخدام المعاملات المرسلة للدالة.
4. تنفيذ Cross Validation باستخدام 3 طيات (3-Fold CV) على بيانات التدريب
   بعد تطبيق تقنية SMOTE لمعالجة عدم توازن الفئات.
5. حساب متوسط وانحراف معيار F1-Score عبر جميع الطيات لتقييم استقرار النموذج.
6. تدريب النموذج النهائي على كامل بيانات التدريب بعد تطبيق SMOTE.
7. إجراء التنبؤات على بيانات التدريب والاختبار.
8. حساب مؤشرات الأداء التالية:
      - Accuracy على بيانات التدريب والاختبار.
      - Precision للفئة الإيجابية.
      - Recall للفئة الإيجابية.
      - F1-Score للفئة الإيجابية.
      - Confusion Matrix.
      - Classification Report.
9. حفظ جميع النتائج والملفات التالية:
      - metrics.json : المقاييس النهائية للتجربة.
      - params.json : معاملات النموذج المستخدمة.
      - cv_scores.json : نتائج Cross Validation.
      - classification_report.txt : التقرير التفصيلي للتصنيف.
      - model.joblib : النموذج المدرب.
      - predictions.csv : التنبؤات الفعلية والمتوقعة على بيانات الاختبار.
10. إرجاع قاموس يحتوي على جميع مؤشرات الأداء الخاصة بالتجربة.

Parameters:
----------
exp_name : str
    اسم التجربة ويُستخدم لإنشاء مجلد مستقل لحفظ النتائج.

params : dict
    معاملات نموذج Random Forest مثل:
    n_estimators, max_depth, min_samples_split وغيرها.

Returns:
-------
dict
    قاموس يحتوي على:
    - نتائج Cross Validation.
    - Accuracy.
    - Precision.
    - Recall.
    - F1-Score.
    - عناصر Confusion Matrix.
    - معاملات النموذج المستخدمة.
"""
def run_experiment_cv3(exp_name, params):
    exp_dir = os.path.join(results_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)

    metrics_file = os.path.join(exp_dir, "metrics.json")
    params_file = os.path.join(exp_dir, "params.json")
    cv_scores_file = os.path.join(exp_dir, "cv_scores.json")
    model_file = os.path.join(exp_dir, "model.joblib")
    preds_file = os.path.join(exp_dir, "predictions.csv")
    report_file = os.path.join(exp_dir, "classification_report.txt")

    if os.path.exists(metrics_file):
        print(f"[SKIP] {exp_name} already completed.")
        with open(metrics_file, "r", encoding="utf-8") as f:
            saved_metrics = json.load(f)
        print(saved_metrics)
        return saved_metrics

    print("\n" + "=" * 90)
    print(f"Training {exp_name}")
    print("Params:", params)
    print("=" * 90)

    model = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params
    )

    # 3-fold CV on SMOTE-adjusted training data
    cv_scores = cross_val_score(
        model,
        X_train_smote,
        y_train_smote,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    )

    print("CV F1 scores:", cv_scores)
    print("Mean CV F1  :", round(float(np.mean(cv_scores)), 6))
    print("Std CV F1   :", round(float(np.std(cv_scores)), 6))

    # fit on full SMOTE-adjusted train
    model.fit(X_train_smote, y_train_smote)

    y_train_pred = model.predict(X_train_smote)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train_smote, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    fraud_precision = precision_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    fraud_recall = recall_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    fraud_f1 = f1_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_test, y_test_pred)
    report_text = classification_report(y_test, y_test_pred, zero_division=0, digits=4)

    result = {
        "experiment": exp_name,
        "cv_f1_mean": float(np.mean(cv_scores)),
        "cv_f1_std": float(np.std(cv_scores)),
        "cv_f1_scores": [float(x) for x in cv_scores],
        "train_accuracy": float(train_acc),
        "test_accuracy": float(test_acc),
        "fraud_precision": float(fraud_precision),
        "fraud_recall": float(fraud_recall),
        "fraud_f1": float(fraud_f1),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        **params
    }

    print("Train Accuracy :", round(train_acc, 6))
    print("Test Accuracy  :", round(test_acc, 6))
    print("Precision      :", round(fraud_precision, 4))
    print("Recall         :", round(fraud_recall, 4))
    print("F1-score       :", round(fraud_f1, 4))
    print("Confusion Matrix:")
    print(cm)

    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    with open(params_file, "w", encoding="utf-8") as f:
        json.dump(params, f, indent=2)

    with open(cv_scores_file, "w", encoding="utf-8") as f:
        json.dump({"cv_f1_scores": [float(x) for x in cv_scores]}, f, indent=2)

    with open(report_file, "w", encoding="utf-8") as f:
        f.write(report_text)

    joblib.dump(model, model_file)

    predictions_df = X_test.copy()
    predictions_df["actual_isFraud"] = y_test.values
    predictions_df["predicted_isFraud"] = y_test_pred
    predictions_df.to_csv(preds_file, index=False)

    print(f"\nSaved in: {exp_dir}")
    return result

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data...
Train shape: (5726358, 6)
Test shape : (636262, 6)

Applying SMOTE to training data only...

Original y_train distribution:
isFraud
0    5718966
1       7392
Name: count, dtype: int64

After SMOTE y_train distribution:
isFraud
0    5718966
1    5718966
Name: count, dtype: int64


In [ ]:
run_experiment_cv3(
    "exp_01",
    {
        "n_estimators": 50,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_01
Params: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.99957287 0.99951834 0.99953879]
Mean CV F1  : 0.999543
Std CV F1   : 2.2e-05
Train Accuracy : 0.999999
Test Accuracy  : 0.999373
Precision      : 0.6813
Recall         : 0.9659
F1-score       : 0.799
Confusion Matrix:
[[635070    371]
 [    28    793]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_01


{'experiment': 'exp_01',
 'cv_f1_mean': 0.9995433336687761,
 'cv_f1_std': 2.2491500808217686e-05,
 'cv_f1_scores': [0.9995728695327497, 0.9995183419426106, 0.9995387895309681],
 'train_accuracy': 0.9999991257160823,
 'test_accuracy': 0.9993728998431464,
 'fraud_precision': 0.6812714776632303,
 'fraud_recall': 0.9658952496954933,
 'fraud_f1': 0.798992443324937,
 'tn': 635070,
 'fp': 371,
 'fn': 28,
 'tp': 793,
 'n_estimators': 50,
 'max_depth': None,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_02",
    {
        "n_estimators": 50,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_02
Params: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.99953696 0.99949764 0.99949686]
Mean CV F1  : 0.99951
Std CV F1   : 1.9e-05
Train Accuracy : 0.999887
Test Accuracy  : 0.999302
Precision      : 0.6549
Recall         : 0.9708
F1-score       : 0.7821
Confusion Matrix:
[[635021    420]
 [    24    797]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_02


{'experiment': 'exp_02',
 'cv_f1_mean': 0.9995104873511891,
 'cv_f1_std': 1.8722074654222025e-05,
 'cv_f1_scores': [0.9995369605629888, 0.9994976392189601, 0.9994968622716184],
 'train_accuracy': 0.9998867802326504,
 'test_accuracy': 0.9993021742615463,
 'fraud_precision': 0.6548890714872637,
 'fraud_recall': 0.9707673568818515,
 'fraud_f1': 0.7821393523061825,
 'tn': 635021,
 'fp': 420,
 'fn': 24,
 'tp': 797,
 'n_estimators': 50,
 'max_depth': None,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_03",
    {
        "n_estimators": 50,
        "max_depth": 10,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_03
Params: {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.99233109 0.99186373 0.99219845]
Mean CV F1  : 0.992131
Std CV F1   : 0.000197
Train Accuracy : 0.991028
Test Accuracy  : 0.983959
Precision      : 0.0739
Recall         : 0.9915
F1-score       : 0.1376
Confusion Matrix:
[[625242  10199]
 [     7    814]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_03


{'experiment': 'exp_03',
 'cv_f1_mean': 0.992131088267343,
 'cv_f1_std': 0.00019665441738539875,
 'cv_f1_scores': [0.9923310876163333, 0.9918637277995945, 0.9921984493861012],
 'train_accuracy': 0.9910276612940171,
 'test_accuracy': 0.9839594380931126,
 'fraud_precision': 0.07391264868791428,
 'fraud_recall': 0.9914738124238733,
 'fraud_f1': 0.13756971438228832,
 'tn': 625242,
 'fp': 10199,
 'fn': 7,
 'tp': 814,
 'n_estimators': 50,
 'max_depth': 10,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_04",
    {
        "n_estimators": 50,
        "max_depth": 10,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_04
Params: {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.9920786  0.99166621 0.99216215]
Mean CV F1  : 0.991969
Std CV F1   : 0.000217
Train Accuracy : 0.991153
Test Accuracy  : 0.984497
Precision      : 0.0763
Recall         : 0.9915
F1-score       : 0.1417
Confusion Matrix:
[[625584   9857]
 [     7    814]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_04


{'experiment': 'exp_04',
 'cv_f1_mean': 0.9919689859872184,
 'cv_f1_std': 0.000216797336802286,
 'cv_f1_scores': [0.992078602531313, 0.9916662063704305, 0.9921621490599114],
 'train_accuracy': 0.9911530336078235,
 'test_accuracy': 0.9844969525132728,
 'fraud_precision': 0.076281510636304,
 'fraud_recall': 0.9914738124238733,
 'fraud_f1': 0.14166376609815523,
 'tn': 625584,
 'fp': 9857,
 'fn': 7,
 'tp': 814,
 'n_estimators': 50,
 'max_depth': 10,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_05",
    {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_05
Params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.9995713  0.99952673 0.99953355]
Mean CV F1  : 0.999544
Std CV F1   : 2e-05
Train Accuracy : 1.0
Test Accuracy  : 0.999365
Precision      : 0.6787
Recall         : 0.9647
F1-score       : 0.7968
Confusion Matrix:
[[635066    375]
 [    29    792]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_05


{'experiment': 'exp_05',
 'cv_f1_mean': 0.9995438607550483,
 'cv_f1_std': 1.959993723958886e-05,
 'cv_f1_scores': [0.9995712983315246, 0.9995267330289217, 0.9995335509046989],
 'train_accuracy': 0.9999999125716083,
 'test_accuracy': 0.9993650414451908,
 'fraud_precision': 0.6786632390745502,
 'fraud_recall': 0.9646772228989038,
 'fraud_f1': 0.7967806841046278,
 'tn': 635066,
 'fp': 375,
 'fn': 29,
 'tp': 792,
 'n_estimators': 100,
 'max_depth': None,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_06",
    {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_06
Params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.99953329 0.99950026 0.99950446]
Mean CV F1  : 0.999513
Std CV F1   : 1.5e-05
Train Accuracy : 0.999891
Test Accuracy  : 0.999294
Precision      : 0.6522
Recall         : 0.9708
F1-score       : 0.7802
Confusion Matrix:
[[635016    425]
 [    24    797]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_06


{'experiment': 'exp_06',
 'cv_f1_mean': 0.9995126738278776,
 'cv_f1_std': 1.4681213518987226e-05,
 'cv_f1_scores': [0.9995332939692142, 0.9995002627160328, 0.9995044647983854],
 'train_accuracy': 0.999891239080631,
 'test_accuracy': 0.9992943158635907,
 'fraud_precision': 0.6522094926350246,
 'fraud_recall': 0.9707673568818515,
 'fraud_f1': 0.7802251590797846,
 'tn': 635016,
 'fp': 425,
 'fn': 24,
 'tp': 797,
 'n_estimators': 100,
 'max_depth': None,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_07",
    {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)

[SKIP] exp_07 already completed.
{'experiment': 'exp_07', 'cv_f1_mean': 0.9918857318957478, 'cv_f1_std': 0.0002919012836610765, 'cv_f1_scores': [0.9919316361009709, 0.9915074923854483, 0.9922180672008242], 'train_accuracy': 0.9915694550378512, 'test_accuracy': 0.9848254335478152, 'fraud_precision': 0.07780539093863506, 'fraud_recall': 0.9914738124238733, 'fraud_f1': 0.14428786670211824, 'tn': 625793, 'fp': 9648, 'fn': 7, 'tp': 814, 'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}


{'experiment': 'exp_07',
 'cv_f1_mean': 0.9918857318957478,
 'cv_f1_std': 0.0002919012836610765,
 'cv_f1_scores': [0.9919316361009709, 0.9915074923854483, 0.9922180672008242],
 'train_accuracy': 0.9915694550378512,
 'test_accuracy': 0.9848254335478152,
 'fraud_precision': 0.07780539093863506,
 'fraud_recall': 0.9914738124238733,
 'fraud_f1': 0.14428786670211824,
 'tn': 625793,
 'fp': 9648,
 'fn': 7,
 'tp': 814,
 'n_estimators': 100,
 'max_depth': 10,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
run_experiment_cv3(
    "exp_08",
    {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
)


Training exp_08
Params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
CV F1 scores: [0.99159882 0.99166186 0.99232164]
Mean CV F1  : 0.991861
Std CV F1   : 0.000327
Train Accuracy : 0.991692
Test Accuracy  : 0.985168
Precision      : 0.0795
Recall         : 0.9915
F1-score       : 0.1471
Confusion Matrix:
[[626011   9430]
 [     7    814]]

Saved in: /content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3/exp_08


{'experiment': 'exp_08',
 'cv_f1_mean': 0.9918607767561448,
 'cv_f1_std': 0.0003268961431028409,
 'cv_f1_scores': [0.9915988233663684, 0.9916618641533108, 0.9923216427487555],
 'train_accuracy': 0.9916915925011619,
 'test_accuracy': 0.9851680596986776,
 'fraud_precision': 0.07946114798906677,
 'fraud_recall': 0.9914738124238733,
 'fraud_f1': 0.14713059195661998,
 'tn': 626011,
 'fp': 9430,
 'fn': 7,
 'tp': 814,
 'n_estimators': 100,
 'max_depth': 10,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

results_dir = "/content/drive/MyDrive/fraud processed data/rf_8_experiments_after_smote_cv3"
experiments = ["exp_01", "exp_02", "exp_03","exp_04", "exp_05", "exp_06","exp_07", "exp_08"]

for exp_name in experiments:
    metrics_file = os.path.join(results_dir, exp_name, "metrics.json")
    preds_file = os.path.join(results_dir, exp_name, "predictions.csv")

    if not os.path.exists(metrics_file) or not os.path.exists(preds_file):
        print(f"{exp_name} not found yet.")
        print("-" * 90)
        continue

    with open(metrics_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    preds_df = pd.read_csv(preds_file)

    y_true = preds_df["actual_isFraud"]
    y_pred = preds_df["predicted_isFraud"]

    cm = np.array([
        [data["tn"], data["fp"]],
        [data["fn"], data["tp"]]
    ])

    report_text = classification_report(
        y_true,
        y_pred,
        digits=4,
        zero_division=0
    )

    print("=" * 90)
    print(exp_name)
    print("=" * 90)
    print("Confusion matrix:")
    print(cm)
    print("\nClassification report:\n")
    print(report_text)

exp_01
Confusion matrix:
[[635070    371]
 [    28    793]]

Classification report:

              precision    recall  f1-score   support

           0     1.0000    0.9994    0.9997    635441
           1     0.6813    0.9659    0.7990       821

    accuracy                         0.9994    636262
   macro avg     0.8406    0.9827    0.8993    636262
weighted avg     0.9995    0.9994    0.9994    636262

exp_02
Confusion matrix:
[[635021    420]
 [    24    797]]

Classification report:

              precision    recall  f1-score   support

           0     1.0000    0.9993    0.9997    635441
           1     0.6549    0.9708    0.7821       821

    accuracy                         0.9993    636262
   macro avg     0.8274    0.9851    0.8909    636262
weighted avg     0.9995    0.9993    0.9994    636262

exp_03
Confusion matrix:
[[625242  10199]
 [     7    814]]

Classification report:

              precision    recall  f1-score   support

           0     1.0000    0.9839   